# Diatomic chain: phonon branches, DOS and heat capacity — Quick start

**Author:** [Jesús García Izquierdo](https://github.com/jesus-garcia-izquierdo) · Repository: [diatomic-chain](https://github.com/jesus-garcia-izquierdo/diatomic-chain)

This notebook runs the main results of the repository in a few seconds:

1. acoustic and optical branches $\omega_\pm(k)$ of the 1D diatomic chain;
2. the closed-form phonon density of states $g(\omega)$;
3. the numerical heat capacity $C_V(T)$ versus the Debye and Einstein models.

Units: $\hbar = k_B = 1$. Frequencies, energies and temperatures share the unit $\sqrt{K/m_1}$.

In [ ]:
# Clone the repository when running on Google Colab (skipped if the package is already importable).
import os, sys
try:
    import diatomic_chain  # noqa: F401
except ImportError:
    if not os.path.exists("diatomic-chain"):
        !git clone -q https://github.com/jesus-garcia-izquierdo/diatomic-chain.git
    sys.path.insert(0, os.path.abspath("diatomic-chain"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from diatomic_chain import ChainParameters
from diatomic_chain import dispersion, dos, thermodynamics, models
from diatomic_chain.plotting import use_style, ROLE_COLORS as RC

use_style()

# Reference chain: m2/m1 = 2, equal springs. ChainParameters is an immutable dataclass.
p = ChainParameters(m1=1.0, m2=2.0, K1=1.0, K2=1.0, a=1.0)
print(p.describe())

## 1. Acoustic and optical branches
Bloch's theorem reduces the $2N$ coupled Newton equations to a $2\times2$ Hermitian eigenproblem $D(k)\,\mathbf e=\omega^2\mathbf e$ for every $k$ of the first Brillouin zone.

In [ ]:
k = dispersion.brillouin_zone(p, 801)
w = dispersion.omega(p, k)                       # shape (801, 2): [acoustic, optical]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k, w[:, 0], color=RC["acoustic"], label="acoustic")
ax.plot(k, w[:, 1], color=RC["optical"], label="optical")
ax.plot(k, p.sound_velocity * np.abs(k), "--", lw=1.2, color=RC["debye"], label=r"$v_s|k|$")
lo, hi = p.zone_boundary_frequencies
ax.axhspan(lo, hi, color="0.9", lw=0)            # forbidden band (gap)
ax.set_xlabel("$k$"); ax.set_ylabel(r"$\omega$"); ax.legend(); plt.show()

## 2. Phonon density of states
Closed form: with $s=\sin^2(ka/2)=m_1m_2\,\omega^2(\sigma/\mu-\omega^2)/(4K_1K_2)$, $\;g(\omega)=\frac{1}{\pi}\frac{|s'(\omega)|}{\sqrt{s(1-s)}}$, normalised to 2 modes per cell.

In [ ]:
wg = np.linspace(1e-6, 1.05 * p.omega_max, 4000)
g = dos.analytic_dos(p, wg)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(wg, np.minimum(g, 6), color=RC["exact"], label="exact $g(\\omega)$")
wD = models.debye_frequency(p)
ax.plot([0, wD, wD], [2 / wD, 2 / wD, 0], "--", color=RC["debye"], label="Debye (flat in 1D)")
ax.set_ylim(0, 4.5); ax.set_xlabel(r"$\omega$"); ax.set_ylabel(r"$g(\omega)$"); ax.legend(); plt.show()
print("normalisation N(inf) =", dos.integrated_dos(p, [10.0])[0])

## 3. Heat capacity: exact vs Debye vs Einstein
Canonical ensemble, sum over modes computed by Gauss–Legendre quadrature over the Brillouin zone.

In [ ]:
T = np.geomspace(1e-3, 5, 300)
C = thermodynamics.heat_capacity(p, T)
C_D = models.debye_heat_capacity(T, models.debye_frequency(p))
C_E = models.einstein_heat_capacity(T, models.einstein_frequency_second_moment(p))
C_H = models.hybrid_heat_capacity(p, T)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(T, C, color=RC["exact"], lw=2.4, label="exact")
ax.loglog(T, C_D, "--", color=RC["debye"], label="Debye")
ax.loglog(T, np.clip(C_E, 1e-6, None), ":", color=RC["einstein"], label="Einstein")
ax.loglog(T, C_H, color=RC["hybrid"], label="Debye + Einstein")
ax.set_ylim(1e-3, 3); ax.set_xlabel("$k_BT$"); ax.set_ylabel(r"$C_V/(N k_B)$"); ax.legend(); plt.show()

print("max relative error  Debye :", np.max(np.abs(C_D - C) / C))
print("max relative error  hybrid:", np.max(np.abs(C_H - C) / C))

For the full analysis (effective Debye temperature, displacement fluctuations, Langevin validation) run `python scripts/make_figures.py` and read the theory documents in `documents_spanish/` and `documents_english/`.